# Deep-Sea Classification: Context Reliance & Depth Degradation
This notebook imports from the project's `.py` modules and runs the full pipeline on Colab's GPU.

# Analysis
Loads saved results from the training pipeline and runs all statistical
analyses: CRS–degradation correlation, baseline comparisons, robustness
checks, bootstrap confidence intervals, and CRS variance analysis.
No GPU required.

In [ ]:
# --- Environment bootstrap (Colab or local; no Google Drive) ---
import os, sys
try:
    import google.colab          # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Pull the code and the saved results/ from the PUBLIC repo — no Drive.
    if not os.path.isdir("cv-project"):
        !git clone --depth 1 https://github.com/kay-asp/cv-project.git
    %cd cv-project
    # Colab already ships torch/numpy/pandas/scipy/matplotlib; add only the extras.
    %pip install -q fathomnet grad-cam pycocotools
# Local: install from requirements.txt (see README) and launch Jupyter from the repo root.

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import os

import importlib
import pandas as pd
import config
import analysis
import extensions

# Load saved results
per_species_df = pd.read_csv(os.path.join(config.RESULTS_DIR, 'results.csv'), index_col='species')

# Reconstruct per-species dicts used by analysis functions
crs_scores   = per_species_df['shallow_crs'].dropna().to_dict()   # CRS on shallow test crops
degradation  = per_species_df['degradation'].dropna().to_dict()
shallow_acc  = per_species_df['shallow_acc'].dropna().to_dict()
uniform_crs  = per_species_df['uniform_crs'].dropna().to_dict()
gaussian_crs = per_species_df['gaussian_crs'].dropna().to_dict()

# Load cached predictions
int_preds = pd.read_csv(os.path.join(config.RESULTS_DIR, 'intervention_predictions.csv'))

all_ic = pd.read_csv(os.path.join(config.RESULTS_DIR, 'image_crs.csv'))
image_crs = {s: all_ic[all_ic['split'] == s].drop(columns='split') for s in ['shallow', 'deep']}

crs_cw = pd.read_csv(os.path.join(config.RESULTS_DIR, 'crs_correct_vs_incorrect.csv'))

# ── Species CRS from correctly classified shallow crops only ──
# Misclassified crops have higher CRS and lower shallow accuracy, so including them links CRS and
# degradation through the same errors; their true-class maps also explain a class the model did not choose.
# The all-crops columns are kept as shallow_all_* and define the qualifying set (139 species).
import attribution
crs_cols = ['shallow_crs', 'shallow_crs_median', 'shallow_crs_chance', 'shallow_crs_excess',
            'shallow_crs_pred', 'shallow_crs_pred_excess', 'shallow_n_crs']
for c in crs_cols:
    per_species_df[c.replace('shallow_', 'shallow_all_', 1)] = per_species_df[c]
shallow_ic = image_crs['shallow']
correct_crs = attribution.species_crs_table(shallow_ic[shallow_ic['correct']],
                                            per_species_df.index.tolist(), prefix='shallow_')
per_species_df[crs_cols] = correct_crs[crs_cols]

crs_scores  = per_species_df['shallow_crs'].dropna().to_dict()
# Uniform noise averages out to a flat map, so its expected CRS is the flat-map CRS (1 - coverage) on the same crops
uniform_crs = per_species_df['shallow_crs_chance'].dropna().to_dict()
# gaussian_crs comes from results.csv: all crops, unless train.ipynb cell 28 was rerun for correct crops

## CRS vs degradation

In [ ]:
# Core hypothesis test - correlation between depth degradation and context relience

# will have to check for a minimum number of test images in deep and shallow
# currently images with 1 test shallow will be weight equally to those with hundereds
# and those with 0 will give crs_score NaN
qualifying = analysis.select_qualifying_species(per_species_df, min_test_n=5, crs_col='shallow_all_crs')
rho, p_value = analysis.spearman_correlation(crs_scores, degradation, species=qualifying)
print(f'Spearman ρ = {rho:.3f}, p = {p_value:.4f}')

summary = analysis.generate_summary_table(
    crs_scores, degradation, rho, p_value, species=qualifying,
    baselines={'Uniform random': uniform_crs, 'Centre Gaussian': gaussian_crs})
summary.to_csv('results/correlation_summary.csv')
summary.round(3)



In [ ]:
analysis.plot_crs_vs_degradation(crs_scores, degradation, species=qualifying, rho=rho,
                                 p_value=p_value, save_path='results/crs_vs_degradation.png')

### CRS correct vs wrong predictions


In [ ]:
display(crs_cw.round(4))
extensions.plot_crs_correct_vs_incorrect(image_crs, species=qualifying,
                                         save_path='results/crs_correct_vs_wrong.png')

### Checking against baselines

In [ ]:
# Checking against baselines
# crs vs organism size (is high crs just because organism is larger in crop?)
analysis.plot_baseline_comparison(crs_scores, uniform_crs, gaussian_crs, species=qualifying,
                                  save_path='results/baseline_crs.png')


In [ ]:
# degradation vs control
# Accuracy vs depth curve for shallow-trained and control models
predictions = pd.read_csv(os.path.join(config.RESULTS_DIR, 'predictions.csv'))
depth_df = pd.read_csv(config.DEPTH_PATH)

analysis.plot_accuracy_vs_depth(
    predictions, depth_df,
    depth_threshold=config.DEPTH_SHALLOW_MAX,
    save_path='results/control_comparison.png')

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1898, in _make_engine
    return mapping[engine](f, **self.options)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/c_parser_wrapper.py", line 93, in __init__
    self._reader = parsers.TextReader(src, **kwds)
                   ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "parsers.pyx", line 574, in pandas._libs.parsers.TextReader.__cinit__
  File "parsers.pyx", line 663, in pandas._libs.parsers.TextReader._get_header
  File "parsers.pyx", line 874, in pandas._libs.parsers.TextReader._tokenize_rows
  File "parsers.pyx", line 891, in pandas._libs.parsers.TextReader._check_tokenize_status
  File "parsers.pyx", line 2053, in pandas._libs.parsers.raise_parser_error
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent ca

In [ ]:
# Context reliance vs depth for both models, true-class and predicted-class CRS
# (the dotted chance line is the flat-map CRS: if CRS follows it, the change is mask size, not attention)
importlib.reload(analysis)
all_ic_ctrl = pd.read_csv(os.path.join(config.RESULTS_DIR, 'image_crs_control.csv'))
crs_by_model = {'shallow': all_ic, 'control': all_ic_ctrl}

for col in ['crs', 'crs_pred']:
    analysis.plot_crs_vs_depth(crs_by_model, depth_df, crs_col=col,
                               depth_threshold=config.DEPTH_SHALLOW_MAX, species=qualifying,
                               save_path=f'results/{col}_vs_depth.png')

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_460/196496492.py", line 3, in <cell line: 0>
    importlib.reload(analysis)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/importlib/__init__.py", line 126, in reload
    spec = module.__spec__ = _bootstrap._find_spec(name, pkgpath, target)
                             ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1265, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1560, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1511, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another 

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_460/196496492.py", line 3, in <cell line: 0>
    importlib.reload(analysis)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/importlib/__init__.py", line 126, in reload
    spec = module.__spec__ = _bootstrap._find_spec(name, pkgpath, target)
                             ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1265, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1560, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1511, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another 

In [ ]:
# Summary table


# Attention maps, shallow-only vs control model on shallow vs deep crops: train.ipynb
# (attribution grid, needs the models) -> results/attribution_grid.png


In [ ]:
# 1. Does the result depend on how many test crops a species needs?
for n in [5, 10, 20]:
    q = analysis.select_qualifying_species(per_species_df, min_test_n=n, crs_col='shallow_all_crs')
    r, p = analysis.spearman_correlation(crs_scores, degradation, species=q)
    print(f"min_test_n={n}: n_species={len(q)}, rho={r:.3f}, p={p:.3f}")

# 2. Species the model barely recognises even in shallow water can't degrade much,
#    which can hide a relationship. Re-run on species it learned reasonably well.
learned = [s for s in qualifying if shallow_acc[s] >= 0.3]
r, p = analysis.spearman_correlation(crs_scores, degradation, species=learned)
print(f"shallow_acc >= 0.3: n_species={len(learned)}, rho={r:.3f}, p={p:.3f}")

# 3. Species CRS from few crops is noisy: resample each species' crops for a CI on ρ
boot_rho = analysis.bootstrap_spearman(shallow_ic[shallow_ic['correct']], degradation, species=qualifying)

# 4. Median instead of mean CRS per species (less swayed by a few extreme crops)
median_crs = per_species_df['shallow_crs_median'].dropna().to_dict()
r, p = analysis.spearman_correlation(median_crs, degradation, species=qualifying)
print(f"median CRS: n_species={len(qualifying)}, rho={r:.3f}, p={p:.3f}")

# Widest per-species CRS intervals (the least certain species), from the train notebook's bootstrap
crs_bootstrap = pd.read_csv(os.path.join(config.RESULTS_DIR, 'crs_bootstrap.csv'), index_col='species')
ci_width = (crs_bootstrap['shallow_crs_ci_high'] - crs_bootstrap['shallow_crs_ci_low']).reindex(qualifying)
display(crs_bootstrap.loc[ci_width.sort_values(ascending=False).index[:10],
                          ['shallow_crs', 'shallow_crs_ci_low', 'shallow_crs_ci_high', 'shallow_n_crs']].round(3))

# 5. Predicted-class CRS: explain the species the model chose rather than the true one (differs only
#    on wrong crops, whose true-class maps are weak and diffuse). Same test, and how close the two are
pred_crs = per_species_df['shallow_crs_pred'].dropna().to_dict()
r, p = analysis.spearman_correlation(pred_crs, degradation, species=qualifying)
print(f"predicted-class CRS: n_species={len(qualifying)}, rho={r:.3f}, p={p:.3f}")
agree = per_species_df.loc[qualifying, ['shallow_crs', 'shallow_crs_pred']].dropna()
print(f"true- vs predicted-class species CRS: Spearman ρ = "
      f"{agree['shallow_crs'].corr(agree['shallow_crs_pred'], method='spearman'):.3f}, "
      f"mean difference (pred − true) = {(agree['shallow_crs_pred'] - agree['shallow_crs']).mean():.3f}")

## Control-adjusted degradation
Some species' deep crops are harder in themselves (darker, smaller, blurrier), and
the control model, which trained on all depths, degrades on those too. Subtracting
its degradation leaves the part caused by the shallow model never seeing deep
water, which is what context reliance is meant to explain:
`degradation_adj = degradation − control_degradation`. The control also had more
training data overall, so it is the best available estimate of intrinsic deep-crop
difficulty rather than an exact "same model, no shift" baseline.

In [ ]:
importlib.reload(analysis)
degradation_adj = (per_species_df['degradation'] - per_species_df['control_degradation']).dropna().to_dict()

# How much of the raw degradation ranking survives the adjustment?
r, p = analysis.spearman_correlation(degradation, degradation_adj, species=qualifying)
print(f"raw vs control-adjusted degradation: ρ = {r:.3f}\n")

adj_rows = []
for name, col in [('CRS', 'shallow_crs'), ('excess CRS', 'shallow_crs_excess'),
                  ('predicted-class CRS', 'shallow_crs_pred')]:
    scores = per_species_df[col].dropna().to_dict()
    for subset, sp in [('qualifying', qualifying), ('shallow_acc >= 0.3', learned)]:
        r, p = analysis.spearman_correlation(scores, degradation_adj, species=sp)
        adj_rows.append({'crs': name, 'species': subset, 'n': len(sp), 'rho': r, 'p': p})
adj_corr = pd.DataFrame(adj_rows)

boot_adj = analysis.bootstrap_spearman(shallow_ic[shallow_ic['correct']], degradation_adj, species=qualifying)
adj_corr.to_csv(os.path.join(config.RESULTS_DIR, 'control_adjusted_correlation.csv'), index=False)
display(adj_corr.round(3))

rho_adj_deg, p_adj_deg = analysis.spearman_correlation(crs_scores, degradation_adj, species=qualifying)
analysis.plot_crs_vs_degradation(crs_scores, degradation_adj, species=qualifying, rho=rho_adj_deg,
                                 p_value=p_adj_deg, ylabel='Control-adjusted degradation',
                                 save_path='results/crs_vs_adjusted_degradation.png')

## Crop-level model: CRS × depth
The species-level tests reduce each species to one degradation value, so a species
with 6 test crops counts as much as one with 600, and every crop's actual depth is
reduced to shallow/deep. Here every test crop is one observation:
`correct ~ depth × crs`, with crs the species' shallow CRS (standardised; never
the crop's own CRS, which the shift itself affects). Crops of one species are not
independent, which is handled two ways: a GEE with standard errors clustered by
species, and a mixed model with a random intercept per species (its own baseline
accuracy). The `depth:crs` coefficient is the test, in log-odds of a correct
prediction per SD of CRS: negative = high-CRS species lose more accuracy at depth.
Fitted with depth as shallow/deep, as continuous km relative to the threshold, and
with the control model's predictions added (`depth:crs:shallow_model`: is the
effect stronger for the shallow-trained model than for the control?).

In [ ]:
importlib.reload(analysis)
specs = {
    'deep vs shallow': {},
    'depth (km)': {'depth_df': depth_df, 'depth_threshold': config.DEPTH_SHALLOW_MAX},
    'excess CRS': {'crs_col': 'shallow_crs_excess'},
    'shallow vs control': {'models': ('shallow', 'control')},
}
fits = []
for name, kw in specs.items():
    print(f"\n=== {name} ===")
    fits.append(analysis.crs_depth_model(predictions, per_species_df, species=qualifying, **kw)
                .assign(spec=name))
crs_depth_fits = pd.concat(fits, ignore_index=True)
crs_depth_fits.to_csv(os.path.join(config.RESULTS_DIR, 'crs_depth_model.csv'), index=False)

key_terms = crs_depth_fits[crs_depth_fits['term'].isin(['depth:crs', 'depth:crs:shallow_model'])]
display(key_terms.set_index(['spec', 'method'])[['term', 'coef', 'ci_low', 'ci_high', 'p']].round(3))

## Background Intervention Results

In [ ]:
importlib.reload(extensions)

int_summary = extensions.intervention_summary(int_preds, species=qualifying)
int_summary.to_csv(os.path.join(config.RESULTS_DIR, 'intervention_summary.csv'))
display(int_summary.round(3))
extensions.plot_intervention_summary(int_summary, save_path='results/intervention_accuracy.png')

# Does greying the background hurt the species Grad-CAM said rely on it?
bg_dep, bg_dep_stats = extensions.background_dependence(int_preds, crs_scores, species=qualifying)




In [ ]:
boot = extensions.bootstrap_intervention(int_preds, species=qualifying)
boot.to_csv(os.path.join(config.RESULTS_DIR, 'intervention_bootstrap.csv'), index=False)
display(boot.round(3))


### Background Swap

In [ ]:
# Background swap: how much of the degradation follows the background vs the organism?
importlib.reload(extensions)
swap_preds = pd.read_csv(os.path.join(config.RESULTS_DIR, 'swap_predictions.csv'))

swap = extensions.swap_decomposition(swap_preds, species=qualifying)
swap.to_csv(os.path.join(config.RESULTS_DIR, 'swap_decomposition.csv'), index=False)
display(swap.round(3))

extensions.plot_swap(swap_preds, species=qualifying, save_path='results/background_swap.png')

### Within-species CRS variance vs degradation
The image-level analysis shows CRS predicts errors within species, but species-level mean CRS doesn't predict degradation. Does CRS *variance* bridge that gap? Species with inconsistent context reliance (high variance) may be more vulnerable to distribution shift than species with stable CRS.

In [ ]:
importlib.reload(extensions)

# Per-species CRS std, IQR, and mean vs degradation (shallow image CRS)
var_df, var_stats = extensions.crs_variance_analysis(
    shallow_ic[shallow_ic['correct']], degradation, species=qualifying)

# Scatter plot for CRS std
rho_std, p_std, n_std = var_stats['std']
extensions.plot_crs_variance_vs_degradation(
    var_df, metric='std', rho=rho_std, p_value=p_std,
    save_path='results/crs_variance_vs_degradation.png')

# Also show IQR (more robust to outliers)
rho_iqr, p_iqr, n_iqr = var_stats['iqr']
extensions.plot_crs_variance_vs_degradation(
    var_df, metric='iqr', rho=rho_iqr, p_value=p_iqr)

display(var_df[['mean', 'std', 'iqr', 'count', 'degradation']].describe().round(3))

### CRS adjusted for mask size
CRS measures the fraction of Grad-CAM activation outside the (dilated) organism mask.
Species with smaller masks will mechanically score higher even if the model
focuses on the organism, because more area lies outside the mask.

Each crop's chance CRS is what a flat heatmap would score on its mask, i.e.
1 − organism coverage (`crs_chance`; the species mean equals `uniform_crs`).
Three ways to remove the geometric effect:
- **Excess CRS** = CRS − chance, averaged per species. 0 means no more
  context-reliant than chance; above 0 means more. It is capped at the
  coverage, so very small organisms can't score highly.
- **Partial Spearman** of CRS vs degradation, controlling for coverage.
- **OLS** `degradation ~ crs + mask_coverage` (standardised predictors).

In [ ]:
importlib.reload(extensions)

adj_df, adj_stats, adj_fit = extensions.crs_mask_size_analysis(per_species_df, species=qualifying)

# Scatter: excess CRS vs degradation
rho_adj, p_adj, n_adj = adj_stats['excess_crs_vs_degradation']
extensions.plot_excess_crs_vs_degradation(
    adj_df, rho=rho_adj, p_value=p_adj,
    save_path='results/excess_crs_vs_degradation.png')

display(adj_df[['crs', 'mask_coverage', 'crs_excess', 'degradation']].describe().round(3))

### Is degradation driven by context reliance?
For species whose accuracy degrades at depth, is this because they relied on
background context in shallow water? If so, removing the background should
shrink the shallow-to-deep accuracy gap for degrading species more than for
non-degrading ones.

In [ ]:
importlib.reload(extensions)
importlib.reload(analysis)


# Shallow model — per-species degradation before and after interventions
ctx_df_grey, _ = extensions.degradation_context_analysis(
    int_preds, degradation, species=qualifying, model='shallow', mode='grey_bg')
ctx_df_blur, _ = extensions.degradation_context_analysis(
    int_preds, degradation, species=qualifying, model='shallow', mode='blur_bg')


# Continuous line plot — shallow model
extensions.plot_degradation_before_after(
    ctx_df_grey, df_blur=ctx_df_blur,
    save_path='results/degradation_before_after.png')

# Binned bar charts
extensions.plot_degradation_bar(
    ctx_df_grey, mode_label='background removed',
    save_path='results/degradation_bar_grey.png')

extensions.plot_degradation_bar(
    ctx_df_blur, mode_label='background blurred',
    save_path='results/degradation_bar_blur.png')

# --- Control model (negative control for regression to the mean) ---
print('\n--- Control model ---')
ctrl_df_grey, _ = extensions.degradation_context_analysis(
    int_preds, degradation, species=qualifying, model='control', mode='grey_bg')
ctrl_df_blur, _ = extensions.degradation_context_analysis(
    int_preds, degradation, species=qualifying, model='control', mode='blur_bg')

extensions.plot_degradation_before_after(
    ctrl_df_grey, df_blur=ctrl_df_blur,
    save_path='results/degradation_before_after_control.png')

extensions.plot_degradation_bar(
    ctrl_df_grey, mode_label='background removed',
    save_path='results/degradation_bar_grey_control.png')

# Accuracy vs depth

analysis.plot_accuracy_vs_depth(
    predictions, depth_df,
    depth_threshold=config.DEPTH_SHALLOW_MAX,
    int_preds=int_preds, int_modes=['grey_bg', 'blur_bg'],
    save_path='results/accuracy_vs_depth_bgintervention.png')

# Species-weighted accuracy vs depth — every species counts equally

# Filter to qualifying species for fair species-weighted comparison
qual_preds = predictions[predictions['species'].isin(qualifying)]
qual_int = int_preds[int_preds['species'].isin(qualifying)]

analysis.plot_accuracy_vs_depth(
    qual_preds, depth_df,
    depth_threshold=config.DEPTH_SHALLOW_MAX,
    int_preds=qual_int, int_modes=['grey_bg', 'blur_bg'],
    species_weighted=True,
    save_path='results/accuracy_vs_depth_species_weighted.png')

### High-degradation species
The overall Spearman looks for one trend across every species, so background
reliance that drives degradation in only some species gets diluted. Here the top
quartile of degradation (fixed in advance; other cutoffs are a sensitivity check
only) is compared with the rest on CRS and background reliance, and quantile
regression tests whether the upper tail of degradation rises with excess CRS.
A species can only degrade as far as its shallow accuracy, so the test is
repeated on relative degradation (degradation / shallow accuracy), and
`shallow_acc` is shown per group as a check.

In [ ]:
importlib.reload(extensions)
hd_df = per_species_df.join(ctx_df_grey['bg_reliance'])
hd_df['relative_degradation'] = hd_df['degradation'] / hd_df['shallow_acc'].where(hd_df['shallow_acc'] > 0)

for deg_col in ['degradation', 'relative_degradation']:
    tbl = extensions.high_degradation_test(hd_df, species=qualifying, deg_col=deg_col)
    tbl.to_csv(os.path.join(config.RESULTS_DIR, f'high_{deg_col}_test.csv'))
    display(tbl.round(3))

# Sensitivity to the cutoff (report, don't pick from)
for q in [0.67, 0.9]:
    t = extensions.high_degradation_test(hd_df, species=qualifying, quantile=q)
    print(t.loc[['shallow_crs_excess', 'bg_reliance'], ['diff', 'p']].round(3))

In [ ]:
# Tail test: does the upper end of degradation rise with excess CRS, even if the middle doesn't?
import statsmodels.formula.api as smf
qr_df = hd_df.loc[qualifying, ['degradation', 'shallow_crs_excess']].dropna()
for q in [0.5, 0.75, 0.9]:
    fit = smf.quantreg('degradation ~ shallow_crs_excess', qr_df).fit(q=q)
    lo, hi = fit.conf_int().loc['shallow_crs_excess']
    print(f"quantile {q}: slope = {fit.params['shallow_crs_excess']:.3f} "
          f"[{lo:.3f}, {hi:.3f}], p = {fit.pvalues['shallow_crs_excess']:.3g}")

## Sensitivity check: same-species masks
Everything above uses the target annotation alone as the organism, so another
individual of the same species in the crop counts as context. Here CRS is
recomputed (train notebook, last section) with every annotation of the
target's species in the crop counted as organism — the pointing-game
convention (Zhang et al. 2016), and a better match for Grad-CAM, which
highlights every individual of the explained class. Only crops with another
annotated individual of the species (`extra_organisms`) can change.

If the conclusions above hold under both mask definitions, they aren't an
artefact of how neighbouring individuals are labelled.

In [ ]:
import matplotlib.pyplot as plt

crs_sp = pd.read_csv(os.path.join(config.RESULTS_DIR, 'crs_same_species.csv'), index_col='species')
all_ic_sp = pd.read_csv(os.path.join(config.RESULTS_DIR, 'image_crs_same_species.csv'))
image_crs_sp = {s: all_ic_sp[all_ic_sp['split'] == s].drop(columns='split') for s in ['shallow', 'deep']}

# Same-species species CRS from correct shallow crops only, like the target-only CRS (cell 3)
sp_correct = attribution.species_crs_table(image_crs_sp['shallow'][image_crs_sp['shallow']['correct']],
                                           crs_sp.index.tolist(), prefix='shallow_')
sp_cols = [c for c in sp_correct.columns if c in crs_sp.columns]
crs_sp[sp_cols] = sp_correct[sp_cols]

# How many crops does the mask definition affect?
for s, ic in image_crs_sp.items():
    q = ic[ic['species'].isin(qualifying)]
    print(f"{s}: {q['extra_organisms'].mean():.1%} of {len(q)} qualifying-species test crops "
          f"contain another individual of the same species")

# Per-species CRS under the two definitions
mask_cmp = pd.DataFrame({'crs_target': per_species_df['shallow_crs'].reindex(qualifying),
                         'crs_same_species': crs_sp['shallow_crs'].reindex(qualifying)}).dropna()
mask_cmp['change'] = mask_cmp['crs_same_species'] - mask_cmp['crs_target']
rho_cmp = mask_cmp['crs_target'].corr(mask_cmp['crs_same_species'], method='spearman')
print(f"Spearman ρ between the two species CRS rankings: {rho_cmp:.3f} (n = {len(mask_cmp)})")
display(mask_cmp.describe().round(3))
display(mask_cmp.sort_values('change').head(10).round(3))   # species whose CRS drops most

fig, ax = plt.subplots(figsize=(3.4, 3.2))
lo, hi = mask_cmp[['crs_target', 'crs_same_species']].min().min(), mask_cmp[['crs_target', 'crs_same_species']].max().max()
ax.plot([lo, hi], [lo, hi], color='#bbbbbb', linewidth=0.8, linestyle='--', zorder=0)
ax.scatter(mask_cmp['crs_target'], mask_cmp['crs_same_species'], s=12, color='#2a78d6', alpha=0.7,
           edgecolor='white', linewidth=0.4)
ax.set_xlabel('CRS, target organism only', fontsize=8)
ax.set_ylabel('CRS, all same-species organisms', fontsize=8)
ax.set_title(f'Species CRS under two mask definitions\nρ = {rho_cmp:.3f}, n = {len(mask_cmp)}', fontsize=8, pad=6)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=7)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig('results/crs_target_vs_same_species.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
importlib.reload(extensions)
importlib.reload(analysis)

# Re-run the main CRS tests under both mask definitions, side by side
versions = {
    'target only': (per_species_df, image_crs),
    'same species': (crs_sp.join(per_species_df['degradation']), image_crs_sp),
}
learned = [s for s in qualifying if shallow_acc[s] >= 0.3]

rows = {}
for name, (sp_df, ic) in versions.items():
    print(f"\n=== {name} ===")
    scores = sp_df['shallow_crs'].dropna().to_dict()
    rho, p = analysis.spearman_correlation(scores, degradation, species=qualifying)
    rho_l, p_l = analysis.spearman_correlation(scores, degradation, species=learned)
    _, adj, _ = extensions.crs_mask_size_analysis(sp_df, species=qualifying)
    _, var = extensions.crs_variance_analysis(ic['shallow'][ic['shallow']['correct']], degradation, species=qualifying)
    cw = {s: extensions.crs_correct_vs_incorrect(ic[s], species=qualifying, label=f'{name}, {s}')
          for s in ['shallow', 'deep']}
    row = {'mean species CRS': sp_df['shallow_crs'].reindex(qualifying).mean(),
           'ρ CRS vs degradation': rho, 'p': p,
           'ρ CRS vs degradation (shallow_acc ≥ 0.3)': rho_l, 'p (shallow_acc ≥ 0.3)': p_l,
           'ρ excess CRS vs degradation': adj['excess_crs_vs_degradation'][0],
           'p excess': adj['excess_crs_vs_degradation'][1],
           'partial ρ CRS vs degradation (coverage)': adj['partial_crs_vs_degradation'][0],
           'p partial': adj['partial_crs_vs_degradation'][1],
           'ρ CRS std vs degradation': var['std'][0], 'p std': var['std'][1]}
    for s in cw:
        row[f'within-species CRS diff, wrong − right ({s})'] = cw[s]['median_within_species_diff']
        row[f'p within-species ({s})'] = cw[s]['p_within']
    rows[name] = row

mask_def_comparison = pd.DataFrame(rows)
mask_def_comparison.to_csv(os.path.join(config.RESULTS_DIR, 'crs_mask_definition_comparison.csv'))
display(mask_def_comparison.round(4))